# 🏆 LLMs vs Traditional ML: Amazon Price Prediction Showdown
**3 Traditional ML Models vs 5 Frontier LLMs — Who Predicts Amazon Prices Best?**

_Powered by OpenRouter · scikit-learn · XGBoost · Gradio_

---

## What I Learned in Week 6

Week 6 covered the full ML pipeline from raw data to fine-tuned frontier models. This notebook extends the key ideas from Days 3 and 4:
- **Day 2** — Using LLMs to pre-process noisy scraped data into clean structured summaries (we load the result from HF)
- **Day 3** — Building an evaluation harness and testing baselines: Linear Regression, Random Forest, XGBoost
- **Day 4** — Zero-shot frontier LLM price prediction; comparing model capabilities out-of-the-box

## Why This Project

The course compared traditional ML vs a couple of frontier LLMs. I wanted to go further and run a **broader, fairer benchmark** across multiple modern frontier models — including open-weight alternatives — to see which truly understands Amazon product pricing best. The original Chinese model candidates were dropped after testing revealed they had limited context on US/Amazon products and performed poorly.

Goals:
1. Run the **same fair benchmark** across 5 diverse frontier LLMs (closed + open-weight)
2. Show that a **single OpenRouter API key** can replace 5 separate provider keys
3. Visualise the results as an interactive **leaderboard** in Gradio

---

## Components

| Component | Purpose |
|---|---|
| `ed-donner/items_lite` (HuggingFace) | Pre-processed Amazon product dataset (~22k items) |
| `scikit-learn` | CountVectorizer, Linear Regression, Random Forest |
| `xgboost` | Gradient-boosted trees baseline |
| `litellm` + OpenRouter | Single API gateway for all 5 LLMs |
| `ThreadPoolExecutor` | Run all 5 LLMs concurrently |
| `gr.Blocks`, `gr.Dataframe`, `gr.BarPlot` | Interactive leaderboard and Challenge Mode |

---

## The 8 Competitors

| # | Group | Model |
|---|---|---|
| 1 | 🏋️ Traditional ML | Linear Regression (Bag-of-Words) |
| 2 | 🏋️ Traditional ML | Random Forest |
| 3 | 🏋️ Traditional ML | XGBoost |
| 4 | 🤖 Frontier LLM | GPT-5.4 |
| 5 | 🤖 Frontier LLM | Claude Sonnet 4.6 |
| 6 | 🤖 Frontier LLM | Gemini 3.1 Flash Lite |
| 7 | 🤖 Frontier LLM | GPT-OSS 120B |
| 8 | 🤖 Frontier LLM | Grok 4.1 Fast |

---

## How It Works

1. **Load** — Pull `ed-donner/items_lite` from HuggingFace Hub (pre-processed product summaries + prices)
2. **Train ML** — Fit 3 classical models on training split (bag-of-words features)
3. **Benchmark** — Run all 8 models on the same 20 test items (LLMs run in parallel)
4. **Score** — Calculate MAE (Mean Absolute Error) per model: lower = better
5. **Explore** — Gradio UI with a sortable leaderboard, grouped bar chart, and Challenge Mode

**Hardware:** Runs entirely on CPU. Requires an OpenRouter API key and a HuggingFace token.

## Cell 1 — Install Dependencies

In [ ]:
# Run once — restart kernel after installation if needed
!uv pip install -q scikit-learn xgboost litellm gradio datasets python-dotenv

print("✅ Dependencies installed.")

## Cell 2 — Imports & Configuration

In [ ]:
import os
import re
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from huggingface_hub import login
import gradio as gr

load_dotenv(override=True)

# ── API Keys ────────────────────────────────────────────────────────────────
hf_token          = os.getenv("HF_TOKEN")
openrouter_key    = os.getenv("OPENROUTER_API_KEY")

if hf_token:
    login(hf_token, add_to_git_credential=True)
    print(f"HuggingFace token: {hf_token[:8]}...")
else:
    print("⚠️  No HF_TOKEN found — dataset loading may fail")

if openrouter_key:
    print(f"OpenRouter key:    {openrouter_key[:12]}...")
else:
    print("⚠️  No OPENROUTER_API_KEY — LLM benchmarking will be skipped")
    print("   Get a free key at https://openrouter.ai")

# ── Benchmark config ────────────────────────────────────────────────────────
BENCHMARK_SIZE = 20       # items from test split for the benchmark
RANDOM_SEED    = 42
random.seed(RANDOM_SEED)

print("\n✅ Configuration ready.")

## Cell 3 — Load Dataset from HuggingFace Hub

In [ ]:
from datasets import load_dataset

# Load Ed Donner's pre-processed lite dataset
print("Loading ed-donner/items_lite from HuggingFace Hub...")

raw = load_dataset("ed-donner/items_lite")

def to_records(split):
    """Convert HF dataset split to list of simple dicts."""
    return [{"summary": r["summary"], "price": float(r["price"])} for r in split]

train_data = to_records(raw["train"])
test_data  = to_records(raw["test"])

print(f"Train items : {len(train_data):,}")
print(f"Test items  : {len(test_data):,}")
print(f"\nSample item:\n  Summary : {train_data[0]['summary'][:120]}...")
print(f"  Price   : ${train_data[0]['price']:.2f}")

## Cell 4 — Evaluation Harness

In [ ]:
def extract_price(text: str) -> float:
    """Pull the first numeric value from an LLM reply string."""
    if isinstance(text, (int, float)):
        return float(text)
    text = str(text).replace(",", "").replace("$", "")
    match = re.search(r"[-+]?\d*\.?\d+", text)
    return float(match.group()) if match else 0.0

def evaluate(pricer_fn, items, label="Model"):
    """
    Run pricer_fn over items and return MAE.
    pricer_fn(item) → price estimate (float or str)
    """
    errors = []
    for item in items:
        try:
            predicted = extract_price(pricer_fn(item))
            actual    = item["price"]
            errors.append(abs(predicted - actual))
        except Exception as e:
            print(f"  [{label}] Error on item: {e}")
            errors.append(None)
    errors = [e for e in errors if e is not None]
    mae = sum(errors) / len(errors) if errors else 9999.0
    return round(mae, 2)

print("✅ Evaluation harness ready.")

## Cell 5 — Train 3 Traditional ML Models

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# ── Prepare data ────────────────────────────────────────────────────────────
train_docs   = [item["summary"] for item in train_data]
train_prices = np.array([item["price"]   for item in train_data])

np.random.seed(RANDOM_SEED)

# CountVectorizer — shared across ML models
print("Fitting CountVectorizer...")
vectorizer = CountVectorizer(max_features=2000, stop_words="english")
X_train    = vectorizer.fit_transform(train_docs)

# ── Linear Regression ───────────────────────────────────────────────────────
print("Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train, train_prices)

# ── Random Forest (subset for speed) ────────────────────────────────────────
SUBSET = 5_000
print(f"Training Random Forest (on {SUBSET:,} items)...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
rf_model.fit(X_train[:SUBSET], train_prices[:SUBSET])

# ── XGBoost ─────────────────────────────────────────────────────────────────
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1, verbosity=0)
xgb_model.fit(X_train, train_prices)

# ── Pricer functions ─────────────────────────────────────────────────────────
def linear_regression_pricer(item):
    x = vectorizer.transform([item["summary"]])
    return max(0.0, lr_model.predict(x)[0])

def random_forest_pricer(item):
    x = vectorizer.transform([item["summary"]])
    return max(0.0, rf_model.predict(x)[0])

def xgboost_pricer(item):
    x = vectorizer.transform([item["summary"]])
    return max(0.0, xgb_model.predict(x)[0])

print("\n✅ All 3 ML models trained.")

## Cell 6 — Define 5 LLM Pricer Functions (via OpenRouter)

In [ ]:
from litellm import completion

# Set OpenRouter API key for litellm
os.environ["OPENROUTER_API_KEY"] = openrouter_key or ""

SYSTEM_PROMPT = """You are a product pricing expert. 
Estimate the price of the product described below. 
Respond with ONLY the dollar amount, e.g. $49.99 — no explanation."""

GPT_OSS_OPTIONS = {
    'model': 'openrouter/openai/gpt-oss-120b',
    "title": "GPT-OSS 120B",
}

GPT_OPTIONS = {
    'model': 'openrouter/openai/gpt-5.4',
    "title": "GPT-5.4",
}
CLAUDE_OPTIONS = {
    'model': 'openrouter/anthropic/claude-sonnet-4.6',
    "title": "Claude Sonnet 4.6",
}
GEMINI_OPTIONS = {
    'model': 'openrouter/google/gemini-3.1-flash-lite-preview',
    "title": "Gemini 3.1 Flash Lite",
}
GROK_OPTIONS = {
    'model': 'openrouter/x-ai/grok-4.1-fast',
    "title": "Grok 4.1 Fast",
}



def _llm_pricer(item, model_id):
    """Generic LLM pricer function — calls any model via OpenRouter."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": item["summary"]}
    ]
    resp = completion(
        model=model_id,
        messages=messages,
        api_base="https://openrouter.ai/api/v1",
        api_key=openrouter_key,
        max_tokens=20,
        verbose=True,
    )
    content = resp.choices[0].message.content
    return (content or "").strip()

# ── US Frontier Models ───────────────────────────────────────────────────────
def gpt_client(item):      return _llm_pricer(item, GPT_OPTIONS.get('model'))
def claude_client(item):  return _llm_pricer(item, CLAUDE_OPTIONS.get('model'))
def gemini_client(item):    return _llm_pricer(item, GEMINI_OPTIONS.get('model'))
def gpt_oss_client(item):     return _llm_pricer(item, GPT_OSS_OPTIONS.get('model'))
def grok_client(item):              return _llm_pricer(item, GROK_OPTIONS.get('model'))

print("✅ All 5 LLM pricer functions defined.")

## Cell 7 — Run the Benchmark

In [ ]:
# Pick a reproducible test subset
random.seed(RANDOM_SEED)
bench_items = random.sample(test_data, min(BENCHMARK_SIZE, len(test_data)))

print(f"Benchmarking {len(bench_items)} items across 8 models...\n")

results = {}   # model_name -> MAE

# ── Traditional ML (sequential) ──────────────────────────────────────────────
ml_models = {
    "Linear Regression" : linear_regression_pricer,
    "Random Forest"     : random_forest_pricer,
    "XGBoost"           : xgboost_pricer,
}
for name, fn in ml_models.items():
    t0  = time.time()
    mae = evaluate(fn, bench_items, label=name)
    elapsed = time.time() - t0
    results[name] = mae
    print(f"  🏋️  {name:<25} MAE = ${mae:.2f}  ({elapsed:.1f}s)")

print()

# ── LLMs (concurrent) ────────────────────────────────────────────────────────
llm_models = {}
if openrouter_key:
    llm_models = {
        f"{GPT_OPTIONS.get('title')}"    : gpt_client,
        f"{CLAUDE_OPTIONS.get('title')}" : claude_client,
        f"{GEMINI_OPTIONS.get('title')}" : gemini_client,
        f"{GPT_OSS_OPTIONS.get('title')}": gpt_oss_client,
        f"{GROK_OPTIONS.get('title')}"   : grok_client,
    }

    def run_llm_benchmark(name, fn):
        t0  = time.time()
        mae = evaluate(fn, bench_items, label=name)
        return name, mae, time.time() - t0

    print("Running all 5 LLMs concurrently (this takes ~30–60 seconds)...")
    t_all = time.time()

    with ThreadPoolExecutor(max_workers=6) as executor:
        futures = {executor.submit(run_llm_benchmark, name, fn): name
                   for name, fn in llm_models.items()}
        for future in as_completed(futures):
            name, mae, elapsed = future.result()
            results[name] = mae
            flag = "🤖"
            print(f"  {flag}  {name:<25} MAE = ${mae:.2f}  ({elapsed:.1f}s)")

    print(f"\nAll LLMs done in {time.time() - t_all:.1f}s total (parallel)")
else:
    print("⚠️  Skipping LLMs — no OPENROUTER_API_KEY found.")

print(f"\n✅ Benchmark complete. {len(results)} models scored.")

## Cell 8 — Gradio Leaderboard UI

In [ ]:
import pandas as pd

def group_label(name):
    if name in ("Linear Regression", "Random Forest", "XGBoost"):
        return "🏋️ Traditional ML"
    else:
        return "🤖 Frontier LLMs"
    

def build_leaderboard_df():
    rows = []
    for rank, (name, mae) in enumerate(
        sorted(results.items(), key=lambda x: x[1]), start=1
    ):
        rows.append({
            "Rank" : rank,
            "Model": name,
            "Group": group_label(name),
            "MAE ($)": f"{mae:.2f}",
            "vs Best": "—" if rank == 1 else f"+${mae - sorted(results.values())[0]:.2f}",
        })
    return pd.DataFrame(rows)

def build_chart_df():
    rows = [{"Model": name, "MAE": mae, "Group": group_label(name)}
            for name, mae in results.items()]
    return pd.DataFrame(sorted(rows, key=lambda x: x["MAE"]))

def challenge_mode():
    """Pick a random test item and show all predictions vs true price."""
    item = random.choice(test_data)
    lines = [f"**Product:** {item['summary'][:200]}",
             f"\n**True Price:** ${item['price']:.2f}",
             "\n---\n**Predictions:**"]

    # ML models
    for name, fn in ml_models.items():
        pred = extract_price(fn(item))
        diff = pred - item["price"]
        lines.append(f"- 🏋️ {name}: **${pred:.2f}** ({'+' if diff >= 0 else ''}{diff:.2f})")

    # LLMs if available
    if openrouter_key:
        with ThreadPoolExecutor(max_workers=6) as ex:
            futures = {ex.submit(fn, item): name for name, fn in llm_models.items()}
            for future in as_completed(futures):
                name  = futures[future]
                pred  = extract_price(future.result())
                diff  = pred - item["price"]
                flag  = "🤖"
                lines.append(f"- {flag} {name}: **${pred:.2f}** ({'+' if diff >= 0 else ''}{diff:.2f})")

    return "\n".join(lines)

# ── Build the Gradio app ─────────────────────────────────────────────────────
with gr.Blocks(title="LLMs vs Traditional ML Showdown") as demo:

    gr.Markdown(
        "# 🏆 LLMs vs Traditional ML Showdown\n"
        "**Traditional ML vs 5 Frontier LLMs — Amazon Price Prediction**\n\n"
        f"_Benchmark: {len(bench_items)} Amazon products · Metric: MAE (lower = better)_"
    )

    with gr.Tabs():

        with gr.Tab("🥇 Leaderboard"):
            gr.Markdown("### Model rankings by Mean Absolute Error (price prediction)")
            leaderboard_df = gr.Dataframe(
                value=build_leaderboard_df(),
                interactive=False,
                wrap=True,
            )

        with gr.Tab("📊 Bar Chart"):
            gr.Markdown("### MAE by Model, grouped by Type")
            chart_df = build_chart_df()
            gr.BarPlot(
                value=chart_df,
                x="Model",
                y="MAE",
                color="Group",
                title="Mean Absolute Error — Lower is Better",
                y_title="MAE ($)",
                x_title="Model",
                height=400,
            )

        with gr.Tab("🎯 Challenge Mode"):
            gr.Markdown(
                "### Pick a random product and see every model's prediction vs the real price!\n"
                "_Click the button to start. Each click draws a new product._"
            )
            challenge_btn    = gr.Button("🎲 Pick a Random Product", variant="primary", size="lg")
            challenge_output = gr.Markdown()
            challenge_btn.click(challenge_mode, outputs=challenge_output)

demo.launch(inbrowser=True)

---

## Concepts Demonstrated

| Cell | Technique | What it shows |
|---|---|---|
| 3 | HF `datasets` | Loading a pre-processed HF Hub dataset in one line |
| 4 | Evaluation harness | Reproduce course `evaluate()` pattern; handles LLM string output |
| 5 | Traditional ML | CountVectorizer → LinearRegression / RandomForest / XGBoost pipeline |
| 6 | LiteLLM + OpenRouter | One API key, five providers, zero per-provider boilerplate |
| 7 | `ThreadPoolExecutor` | Parallel LLM calls — the same `workers` trick used in course `evaluate()` |
| 8 | Gradio `gr.Dataframe` + `gr.BarPlot` | Live leaderboard + grouped visualisation |
| 8 | Challenge Mode | Interactive single-item deep-dive across all 8 models |

---

## Extensions Possible

1. **Add more open-weight models** — DeepSeek-V3, Qwen-72B, Llama 3.3 — all available on OpenRouter
2. **Latency leaderboard** — track tokens/second per model alongside MAE
3. **Cost leaderboard** — use OpenRouter's `response_cost` to show $ per 20 items
4. **Fine-tuned model** — load your own fine-tuned endpoint and add it as a 9th competitor
5. **Category filter** — filter bench items by category (Electronics, Appliances, etc.) to see which model excels where
